# A04 · Backpropagation & Autograd

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a04_backprop.ipynb)

Backprop is the chain rule applied in reverse over a graph, reusing intermediate results so all gradients cost about as much as one extra forward pass. You'll compute it by hand with numbers, then build a tiny autograd engine and derive matrix gradients for an MLP.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a04). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Run forward and backward passes by hand on a small network with real numbers.
- Build a scalar autograd engine (micrograd-style) with topological sort.
- Derive matrix gradients: $\nabla_W = X^\top \delta$, $\nabla_X = \delta W^\top$.
- Verify gradients with finite differences.

## The math

### Forward pass with numbers

$$a = w_1x + b_1,\quad h = \mathrm{ReLU}(a),\quad \hat y = w_2 h + b_2,\quad L = \tfrac12(\hat y - y)^2$$

$x = 2,\ y = 3,\ w_1 = 0.5,\ b_1 = 0,\ w_2 = 2.0,\ b_2 = 0.5$.

**Worked example**

1. $a = 0.5\cdot 2 + 0 = 1.0,\quad h = 1.0$
2. $\hat y = 2.0 \cdot 1.0 + 0.5 = 2.5$
3. $L = \tfrac12(2.5 - 3)^2 = 0.125$

### Backward pass with numbers

$$\frac{\partial L}{\partial w_1} = \underbrace{(\hat y - y)}_{\partial L/\partial \hat y}\cdot \underbrace{w_2}_{\partial \hat y/\partial h}\cdot \underbrace{\mathbb{1}[a>0]}_{\partial h/\partial a}\cdot \underbrace{x}_{\partial a/\partial w_1}$$

Walk from the loss back to each parameter, multiplying local derivatives.

**Worked example**

1. $\partial L/\partial \hat y = 2.5 - 3 = -0.5$
2. $\partial L/\partial w_2 = -0.5\cdot h = -0.5,\quad \partial L/\partial b_2 = -0.5$
3. $\partial L/\partial h = -0.5\cdot w_2 = -1.0 \Rightarrow \partial L/\partial a = -1.0$ (ReLU passes the gradient because a > 0)
4. $\partial L/\partial w_1 = -1.0\cdot x = -2.0,\quad \partial L/\partial b_1 = -1.0$
5. Step $\eta = 0.05$: $w_1 = 0.6,\ b_1 = 0.05,\ w_2 = 2.025,\ b_2 = 0.525$
6. New $\hat y = 3.056$, $L = 0.0016$ (loss fell by about 80×. With $\eta = 0.1$ it overshoots to $L = 0.195$)

### Matrix gradients for a linear layer

$$Y = XW \;\Rightarrow\; \frac{\partial L}{\partial W} = X^{\top}\,\frac{\partial L}{\partial Y}, \qquad \frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y}\,W^{\top}$$

Shapes check it: $X$ is $(B, d_{in})$, $\delta = \partial L/\partial Y$ is $(B, d_{out})$, so $X^\top\delta$ is $(d_{in}, d_{out})$ like $W$. The batch dimension is summed over.

**Worked example**

1. MSE $L = \frac{1}{B}\sum (P - Y)^2 \Rightarrow \partial L/\partial P = \frac{2}{B}(P - Y)$
2. ReLU: $\partial L/\partial A = \partial L/\partial H \odot \mathbb{1}[A > 0]$ (elementwise, written $\odot$)
3. Backward cost ≈ 2× forward FLOPs (two matmuls per linear layer) (hence training ≈ $3\times$ forward, the "6N" rule you'll meet in A09)

## Lab: Build an autograd engine, then backprop an MLP by hand

By the end you'll have written the core of what `loss.backward()` does.

### Step 1

A scalar `Value` class that records the graph and knows each op's local derivative.

In [ ]:
class Value:
    def __init__(self, data, parents=(), op=''):
        self.data, self.grad = data, 0.0
        self._parents, self._op = parents, op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad              # add copies the gradient to both inputs
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad  # multiply swaps the inputs
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data ** k, (self,), f'**{k}')
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad   # gate
        out._backward = _backward
        return out

    __radd__ = __add__
    def __rmul__(self, other): return self * other
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)

    def backward(self):
        order, seen = [], set()
        def build(v):                          # topological order: parents before children
            if v not in seen:
                seen.add(v)
                for p in v._parents: build(p)
                order.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

### Step 2

Reproduce the worked example: you should get exactly the numbers above.

In [ ]:
x, y = Value(2.0), Value(3.0)
w1, b1, w2, b2 = Value(0.5), Value(0.0), Value(2.0), Value(0.5)
h = (w1 * x + b1).relu()
y_hat = w2 * h + b2
L = 0.5 * (y_hat - y) ** 2
L.backward()
print(L.data)                                   # 0.125
print(w1.grad, b1.grad, w2.grad, b2.grad)       # -2.0 -1.0 -0.5 -0.5

### Step 3

Check against PyTorch autograd and a finite-difference estimate.

In [ ]:
import torch
p = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in
     dict(w1=0.5, b1=0.0, w2=2.0, b2=0.5).items()}
def loss_fn(p):
    h = torch.relu(p['w1'] * 2.0 + p['b1'])
    return 0.5 * (p['w2'] * h + p['b2'] - 3.0) ** 2
loss_fn(p).backward()
print({k: v.grad.item() for k, v in p.items()})
eps = 1e-6
with torch.no_grad():
    p['w1'] += eps; up = loss_fn(p)
    p['w1'] -= 2 * eps; down = loss_fn(p)
    p['w1'] += eps
print("finite difference dL/dw1:", ((up - down) / (2 * eps)).item())   # ~ -2.0

### Step 4

Matrix backprop for a 2-layer MLP, by hand, then compare with autograd.

In [ ]:
torch.manual_seed(0)
X, Y = torch.randn(64, 10), torch.randn(64, 1)
W1, W2 = torch.randn(10, 32) * 0.1, torch.randn(32, 1) * 0.1
# forward
A = X @ W1
H = A.clamp(min=0)
P = H @ W2
loss = ((P - Y) ** 2).mean()
# backward by hand
dP = 2 * (P - Y) / P.numel()
dW2 = H.T @ dP            # (32, 64) @ (64, 1)
dH = dP @ W2.T            # (64, 1) @ (1, 32)
dA = dH * (A > 0)         # ReLU gate
dW1 = X.T @ dA            # (10, 64) @ (64, 32)
# autograd
W1a, W2a = W1.clone().requires_grad_(), W2.clone().requires_grad_()
((X @ W1a).clamp(min=0) @ W2a - Y).pow(2).mean().backward()
print(torch.allclose(dW1, W1a.grad, atol=1e-6), torch.allclose(dW2, W2a.grad, atol=1e-6))   # True True

### Clean up

Nothing to clean up.

## Check yourself

**1. $L = (w x)^2$ with $w = 3,\ x = 2$. What is $\partial L/\partial w$?**

A. 12  
B. 24  
C. 36  
D. 6

<details><summary>Answer</summary>

**B.** $\partial L/\partial w = 2(wx)\cdot x = 2\cdot6\cdot2 = 24$.

</details>

**2. A variable feeds two branches whose upstream gradients are $+0.3$ and $-0.5$. Its gradient?**

A. $-0.5$  
B. $-0.2$  
C. $0.3$  
D. $-0.15$

<details><summary>Answer</summary>

**B.** Gradients from multiple uses add (multivariate chain rule): 0.3 + (−0.5) = −0.2.

</details>

**3. $X$ is $(B, 512)$, $W$ is $(512, 2048)$, $\delta = \partial L/\partial Y$ is $(B, 2048)$. Which expression has $W$'s shape?**

A. $\delta W^\top$  
B. $X^\top\delta$  
C. $\delta^\top X$  
D. $W\delta^\top$

<details><summary>Answer</summary>

**B.** $X^\top\delta$ is $(512, B)(B, 2048) = (512, 2048)$. $\delta^\top X$ gives the transpose.

</details>